# Temporal conformal tabular: predictions

This notebook trains TabPFN-2 and gradient boosting on three public time-stamped datasets, once with a random split
and once with a time-ordered split. It saves the 5%, 50% and 95% predicted quantiles for the calibration
and test rows.

Built with PriorLabs-TabPFN.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

Results are saved to Google Drive as the run progresses. If Colab disconnects, run all cells again and the
notebook continues from where it stopped.

In [ ]:
DATASETS = ["bike", "metro", "pm25"]
SPLITS = ["random", "temporal"]
MODELS = ["gbdt", "tabpfn"]      # the faster model runs first
MAX_TRAIN_ROWS = 5000            # both models train on the same rows
TRAIN_SHARE = 0.5
CALIBRATION_SHARE = 0.1
N_ESTIMATORS = 4                 # TabPFN ensemble size
CHUNK_ROWS = 1000                # TabPFN predicts this many rows at a time
LIMIT_ROWS = None                # set a number only for a quick test
OUTPUT_FOLDER = "temporal-conformal-tabular"


In [ ]:
import subprocess, sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn==9.2.0"], check=True)


In [ ]:
import io, os, shutil, time, zipfile
from pathlib import Path

os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
import numpy as np
import pandas as pd
import requests
import torch
from huggingface_hub import hf_hub_download
from sklearn.ensemble import HistGradientBoostingRegressor
from tabpfn import TabPFNRegressor

ALLOW_CPU = globals().get("ALLOW_CPU", False)
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
elif ALLOW_CPU:
    DEVICE = "cpu"
else:
    raise RuntimeError("No GPU found. Choose Runtime > Change runtime type > T4 GPU, then run all cells again.")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive") / OUTPUT_FOLDER / "predictions"
else:
    OUTPUT_DIR = Path(globals().get("LOCAL_OUTPUT_DIR", OUTPUT_FOLDER)) / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to", OUTPUT_DIR)


In [ ]:
UCI = "https://archive.ics.uci.edu/static/public/{}.zip"
DATA_DIR = Path(globals().get("LOCAL_DATA_DIR", "uci"))
DATA_DIR.mkdir(exist_ok=True)

def read_from_zip(name, slug, member):
    path = DATA_DIR / f"{name}.zip"
    if not path.exists():
        response = requests.get(UCI.format(slug), timeout=180)
        response.raise_for_status()
        path.write_bytes(response.content)
    with zipfile.ZipFile(path) as archive:
        return pd.read_csv(io.BytesIO(archive.read(member)), compression="gzip" if member.endswith(".gz") else None)

def codes(series):
    return series.astype("category").cat.codes

def load_bike():
    df = read_from_zip("bike", "275/bike+sharing+dataset", "hour.csv")
    df["time"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(df["hr"], unit="h")
    features = ["season", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]
    return df.sort_values("time").reset_index(drop=True), features, "cnt"

def load_metro():
    df = read_from_zip("metro", "492/metro+interstate+traffic+volume", "Metro_Interstate_Traffic_Volume.csv.gz")
    df["time"] = pd.to_datetime(df["date_time"])
    # The file has several rows for one hour when more than one weather type was recorded.
    df = df.sort_values("time").drop_duplicates("time").reset_index(drop=True)
    df["month"], df["hour"], df["weekday"] = df["time"].dt.month, df["time"].dt.hour, df["time"].dt.dayofweek
    df["is_holiday"] = df["holiday"].fillna("None").ne("None").astype(int)
    df["weather"] = codes(df["weather_main"])
    features = ["month", "hour", "weekday", "is_holiday", "temp", "rain_1h", "snow_1h", "clouds_all", "weather"]
    return df, features, "traffic_volume"

def load_pm25():
    df = read_from_zip("pm25", "381/beijing+pm2+5+data", "PRSA_data_2010.1.1-2014.12.31.csv")
    df = df.dropna(subset=["pm2.5"]).copy()
    df["time"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
    df["weekday"] = df["time"].dt.dayofweek
    df["wind_direction"] = codes(df["cbwd"])
    features = ["month", "hour", "weekday", "DEWP", "TEMP", "PRES", "wind_direction", "Iws", "Is", "Ir"]
    return df.sort_values("time").reset_index(drop=True), features, "pm2.5"

LOADERS = {"bike": load_bike, "metro": load_metro, "pm25": load_pm25}

def load_dataset(name):
    df, features, target = LOADERS[name]()
    if LIMIT_ROWS:
        df = df.iloc[np.linspace(0, len(df) - 1, LIMIT_ROWS).astype(int)].reset_index(drop=True)
    X = df[features].to_numpy(dtype=np.float32)
    y = df[target].to_numpy(dtype=np.float32)
    assert np.isfinite(X).all() and np.isfinite(y).all(), f"{name} has missing values"
    assert df["time"].is_monotonic_increasing, f"{name} is not in time order"
    return df["time"], X, y

def make_split(n, split):
    """Return row positions for training, calibration and test, each in time order."""
    positions = np.arange(n) if split == "temporal" else np.random.default_rng(0).permutation(n)
    n_train, n_calibration = int(TRAIN_SHARE * n), int(CALIBRATION_SHARE * n)
    train = positions[:n_train]
    if len(train) > MAX_TRAIN_ROWS:
        train = np.random.default_rng(1).choice(train, MAX_TRAIN_ROWS, replace=False)
    return np.sort(train), np.sort(positions[n_train : n_train + n_calibration]), np.sort(positions[n_train + n_calibration :])

for name in DATASETS:
    times, X, y = load_dataset(name)
    print(f"{name}: {len(y)} rows, {X.shape[1]} features, {times.iloc[0].date()} to {times.iloc[-1].date()}")


In [ ]:
WEIGHTS = hf_hub_download("Prior-Labs/TabPFN-v2-reg", "tabpfn-v2-regressor-v2_default.ckpt")
QUANTILES = [0.05, 0.5, 0.95]

def predict_gbdt(X_train, y_train, X_rows):
    columns = []
    for quantile in QUANTILES:
        model = HistGradientBoostingRegressor(
            loss="quantile", quantile=quantile, learning_rate=0.05, max_iter=300,
            max_leaf_nodes=31, min_samples_leaf=20, early_stopping=False, random_state=0,
        )
        columns.append(model.fit(X_train, y_train).predict(X_rows))
    return np.column_stack(columns)

def tabpfn_model(X_train, y_train):
    model = TabPFNRegressor(
        model_path=WEIGHTS, device=DEVICE, n_estimators=N_ESTIMATORS,
        ignore_pretraining_limits=True, random_state=0,
    )
    return model.fit(X_train, y_train)

def predict_tabpfn_chunk(model, X_chunk):
    return np.column_stack(model.predict(X_chunk, output_type="quantiles", quantiles=QUANTILES))

def clean_quantiles(values):
    assert np.isfinite(values).all(), "The model returned NaN or infinity."
    return np.sort(values, axis=1)  # separate quantile models can cross, so keep them in order


In [ ]:
# A small check before the long run: both models must give finite, ordered, sensible quantiles.
times, X, y = load_dataset(DATASETS[0])
train, calibration, test = make_split(len(y), "temporal")
few = train[:: max(len(train) // 300, 1)]
rows = calibration[:100]
for name, values in {
    "gbdt": predict_gbdt(X[few], y[few], X[rows]),
    "tabpfn": predict_tabpfn_chunk(tabpfn_model(X[few], y[few]), X[rows]),
}.items():
    values = clean_quantiles(values)
    error = np.abs(y[rows] - values[:, 1]).mean()
    assert error < np.abs(y[rows] - y[few].mean()).mean() * 1.5, f"{name} error looks wrong: {error:.2f}"
    print(f"{name}: median error {error:.2f}, band width {(values[:, 2] - values[:, 0]).mean():.2f}")
print("Self-check passed.")


In [ ]:
def write_final(part, final):
    frame = pd.read_csv(part)
    frame.to_csv(final, index=False, compression="gzip", float_format="%.4f")
    part.unlink()

started = time.time()
for dataset in DATASETS:
    times, X, y = load_dataset(dataset)
    for split in SPLITS:
        train, calibration, test = make_split(len(y), split)
        positions = np.concatenate([calibration, test])
        roles = np.array(["calibration"] * len(calibration) + ["test"] * len(test))
        for model_name in MODELS:
            final = OUTPUT_DIR / f"{dataset}__{split}__{model_name}.csv.gz"
            if final.exists():
                print("already done:", final.name)
                continue
            part = OUTPUT_DIR / f"{final.name}.part.csv"
            done = len(pd.read_csv(part)) if part.exists() else 0
            print(f"{final.name}: {len(train)} train rows, {len(positions)} rows to predict, {done} already saved", flush=True)
            if model_name == "gbdt":
                model = None
                step = len(positions)
            else:
                model = tabpfn_model(X[train], y[train])
                step = CHUNK_ROWS
            for start in range(done, len(positions), step):
                chunk = positions[start : start + step]
                if model is None:
                    values = predict_gbdt(X[train], y[train], X[chunk])
                else:
                    values = predict_tabpfn_chunk(model, X[chunk])
                values = clean_quantiles(values)
                frame = pd.DataFrame({
                    "position": chunk,
                    "time": times.iloc[chunk].dt.strftime("%Y-%m-%d %H:%M").to_numpy(),
                    "role": roles[start : start + step],
                    "y": y[chunk],
                    "q05": values[:, 0],
                    "q50": values[:, 1],
                    "q95": values[:, 2],
                })
                frame.to_csv(part, mode="a", header=not part.exists(), index=False, float_format="%.4f")
                print(f"  {min(start + step, len(positions))}/{len(positions)} rows | {(time.time() - started) / 60:.1f} min so far", flush=True)
            write_final(part, final)
print("All predictions finished.")


In [ ]:
# A quick check only. The conformal analysis runs later on the laptop with src/analyze.py.
for path in sorted(OUTPUT_DIR.glob("*.csv.gz")):
    frame = pd.read_csv(path)
    test = frame[frame["role"] == "test"]
    band = ((test["y"] >= test["q05"]) & (test["y"] <= test["q95"])).mean()
    print(f"{path.name:<40} rows {len(frame):>6} | median error {(test['y'] - test['q50']).abs().mean():>8.2f} | raw band coverage {band:.3f}")
print("\nDownload the whole folder from Google Drive and put its files in results/predictions/ on your laptop:")
print(OUTPUT_DIR)
